# Feature Engineering

After EDA, I have a clear picture of what I'm working with. 
180K orders, 23 columns, no nulls and saved to data_clean.csv. 
The big takeaway from EDA was that Shipping Mode is doing most of the 
heavy lifting as a predictor. First Class sitting at 95% late rate was 
a surprise. Market and Order Status were basically flat 
across categories so I'm not expecting much from those.

Now I need to get this dataset into a form the model can actually use. 
That means extracting signal from the date column, handling the skewed 
columns, encoding the cat variables, and building a couple of new features 
that might capture something.

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
#load df_clean and convert to pandas df
df = spark.read.table("novasend_fulfillment.processed.data_clean").toPandas()

print(f"Shape: {df.shape}")
print(f"Nulls: {df.isnull().sum().sum()}")
print(df.dtypes)

In [0]:
# Replace spaces in column names with underscores — Spark does not allow
# spaces in Delta table column names
df.columns = [col.replace(" ", "_") for col in df.columns]

# Confirm the problematic columns are fixed
print([c for c in df.columns if " " in c])  # should print empty list

## Temporal Features

I'm going to change the current Order Date column to pull out hthe following
1. day of 
2. week
3. month

I'd expect orders placed on Fridays or heading into peak months to have
a different late rate than orders placed mid-week in slow periods.
Weekend orders might also behave differently if warehouse staffing drops on Saturdays and Sundays.

Going to extract day of week, month, quarter, and a simple is_weekend 
flag then drop the raw date column entirely.

In [0]:
# Parse the order date column: coerce errors to NaT so bad rows don't crash
df["order_date_DateOrders"] = pd.to_datetime(
    df["order_date_DateOrders"], infer_datetime_format=True, errors="coerce"
)

# Confirm no NaT values after parsing — if this prints > 0, investigate
print(f"Unparseable dates: {df['order_date_DateOrders'].isnull().sum()}")

# Extract temporal components — these capture seasonality and workload patterns
df["order_day_of_week"] = df["order_date_DateOrders"].dt.dayofweek   # 0=Monday, 6=Sunday
df["order_month"]       = df["order_date_DateOrders"].dt.month        # 1–12
df["order_quarter"]     = df["order_date_DateOrders"].dt.quarter      # 1–4
df["is_weekend"]        = df["order_date_DateOrders"].dt.dayofweek.isin([5, 6]).astype(int)

# Drop the raw date column
df = df.drop(columns=["order_date_DateOrders"])

# Quick sanity check on the new features
print(df[["order_day_of_week", "order_month", "order_quarter", "is_weekend"]].describe())

## Log Transformations

Parsing worked cleanly, no NaT values. The temporal features look 
reasonable. month covers 1 through 12, is_weekend is binary, nothing 
unexpected.

EDA flagged Sales per customer, Order Item Product Price, and Order Item 
Discount as heavily right-skewed. For logistic regression 
those outliers are going to cause problems. LightGBM and XGBoost handle 
skew better natively but I want consistent features going into all three 
models. Log transforming with np.log1p so zero values don't blow up.

In [0]:
# Columns identified in EDA as right-skewed — large outliers will dominate
# tree-based models less than linear ones, but log scaling still helps
# with split quality and keeps feature magnitudes comparable
skewed_cols = [
    "Sales_per_customer",
    "Order_Item_Product_Price",
    "Order_Item_Discount",
]

for col in skewed_cols:
    # np.log1p handles zero values safely — log(0) is undefined, log(1+0) = 0
    new_col = "log_" + col.lower()
    df[new_col] = np.log1p(df[col])

# Drop the raw skewed columns — the log versions replace them
df = df.drop(columns=skewed_cols)

# Confirm the new columns exist and check their distributions look reasonable
print(df[[
    "log_sales_per_customer",
    "log_order_item_product_price",
    "log_order_item_discount"
]].describe().round(3))

## Discount Aggressiveness Tier

Log distributions look good. Means and ranges are all sensible, nothing 
suspicious. log_order_item_discount has a min of 0 which makes sense — 
plenty of orders have no discount applied at all.

Order Item Discount Rate is already in the dataset as a continuous value 
but I want to also capture it as a categorical tier. The idea is that 
the relationship between discount depth and late risk probably isn't 
linear. There might be a threshold effect where aggressive discounting 
signals something about the order that correlates with fulfillment 
problems. Bucketing into none, low, medium, aggressive to test that.

In [0]:
# Order_Item_Discount_Rate is a continuous value between 0 and 1
# Bucketing it into tiers makes the signal more interpretable for the model
# and captures non-linear relationships between discount depth and late risk

discount_bins   = [-0.01, 0.0, 0.1, 0.2, 1.0]
discount_labels = ["none", "low", "medium", "aggressive"]

df["discount_tier"] = pd.cut(
    df["Order_Item_Discount_Rate"],
    bins=discount_bins,
    labels=discount_labels
)

# Confirm the distribution across tiers — we want all four populated
print(df["discount_tier"].value_counts().sort_index())
print(f"\nNulls: {df['discount_tier'].isnull().sum()}")

## Target Encoding — Order Region

Tier distribution came out well. All four buckets are populated above 
10K which means no sparse categories that would make the encoding 
unreliable. Low and medium are the dominant tiers which makes sense 
given how the discount rates were distributed in EDA.

Order Region has 20+ unique values. One-hot encoding that would create 
a messy wide feature space and throw away the relationship between 
region and late risk. Target encoding replaces each region with its 
historical late delivery rate which is a much more direct signal.

The catch is leakage. If I just compute global late rates per region 
and map them in, each row is partly encoded using itself. Using KFold 
cross-validation so every row gets encoded from a fold it wasn't part of.

In [0]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

df["region_late_rate"] = 0.0  # initialize the new column

for train_idx, val_idx in kf.split(df):
    # Compute late rate per region using only the training fold
    region_means = (
        df.iloc[train_idx]
        .groupby("Order_Region")["Late_delivery_risk"]
        .mean()
    )
    # Map those rates onto the validation fold rows only
    df.loc[df.index[val_idx], "region_late_rate"] = (
        df.iloc[val_idx]["Order_Region"].map(region_means)
    )

print(f"Nulls: {df['region_late_rate'].isnull().sum()}")
print(df["region_late_rate"].describe().round(3))

## Encoding Remaining Categoricals

That region output is telling me something important. The std is 0.010 
and the full range is only 0.48 to 0.584 — every region is clustering 
within about 10 percentage points of the global mean. Same story as 
Market in EDA. Geography just isn't differentiating late risk in this 
dataset the way I expected it to.

I'm keeping region_late_rate in since it cost nothing and SHAP will 
confirm whether it's worth anything. But I'm scrapping the Shipping Mode 
× Order Region interaction feature I had planned — if region is nearly 
flat, the interaction isn't going to add signal on top of Shipping Mode 
standing alone.

Now I need to one-hot encode the remaining low-cardinality categoricals 
and drop the high-cardinality columns that don't have a clean encoding 
strategy at this stage.

In [0]:
# Drop Order_Region — region_late_rate is its encoded replacement
df = df.drop(columns=["Order_Region"])

# These low-cardinality categoricals get one-hot encoded
# drop_first=True drops one dummy per feature to avoid multicollinearity
ohe_cols = [
    "Type",
    "Customer_Segment",
    "Market",
    "Order_Status",
    "Shipping_Mode",
    "discount_tier",
]

df = pd.get_dummies(df, columns=ohe_cols, drop_first=True)

# High-cardinality columns dropped entirely
df = df.drop(columns=[
    "Customer_City",
    "Customer_Country",
    "Customer_State",
    "Order_City",
    "Order_Country",
    "Order_State",
    "Category_Name",
    "Department_Name",
    "Product_Name",
])

print(f"Shape: {df.shape}")
print(df.dtypes.value_counts())

## Dtype Cleanup

35 features, 180K rows — that's a clean lean feature set. One thing I 
need to fix before saving is that pd.get_dummies produces bool dtype in 
pandas 2.x. Some versions of sklearn and LightGBM handle that fine but 
it's cleaner to cast everything to int8 explicitly. Smaller memory 
footprint too.

In [0]:
# Cast bool columns to int32 — Spark's Arrow converter does not support int8
# int32 is the safe default for Spark compatibility
bool_cols = df.select_dtypes(include="uint8").columns.tolist()
df[bool_cols] = df[bool_cols].astype("int32")

# Confirm no bool columns remain
print(df.dtypes.value_counts())

In [0]:
# Convert Pandas DataFrame back to Spark to write as a Delta table
spark_df = spark.createDataFrame(df)

# Write to the processed schema — overwrite replaces on each rerun
spark_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("novasend_fulfillment.processed.features_engineered")

print("Saved to: novasend_fulfillment.processed.features_engineered")
print(f"Shape: {df.shape}")
print(f"Nulls: {df.isnull().sum().sum()}")